# Stage 2 v4 — Recovery: Build v8 + Push HF (sau kernel restart)

**Dùng khi:** Groq batch đã chạy xong (186 batches DONE, output files trong `output_aug_v3/`),
nhưng kernel bị restart làm mất biến `v8_rows`.

**Pipeline (chỉ các bước còn lại):**
1. Load `items_tv_v7` từ HF Hub (đã push, skip re-merge)
2. Filter + parse_summary → `train_filtered`
3. Bucket multipliers → `bucket_multipliers`
4. Define `AugBatchManager` + load state từ `batches_aug_v3.pkl`
5. Đọc output files local → rebuild `aug_results`
6. Build `v8_rows` → push `items_tv_v8`
7. Build prompts → push `items_prompts_tv_4`

**Không cần GPU. Không cần Groq API.**

In [1]:
import os
import re
import json
import time
import pickle
import random
import numpy as np
from pathlib import Path
from tqdm.auto import tqdm
from dataclasses import dataclass
from typing import Optional

from datasets import load_dataset, DatasetDict, Dataset
from dotenv import load_dotenv
from huggingface_hub import login

NOTEBOOK_DIR = Path(".")
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

# --- HF datasets ---
SOURCE_TV7  = "SeanSunny/items_tv_v7"
OUTPUT_TV8  = "SeanSunny/items_tv_v8"
SOURCE_TV3  = "SeanSunny/items_prompts_tv_3"
OUTPUT_TV4  = "SeanSunny/items_prompts_tv_4"

MAX_PRICE     = 1_000_000
QUESTION_FULL = "Sản phẩm này có giá bao nhiêu ?"
PRICE_PREF    = "\n\nGiá là: "

# --- Folders (dùng chung với v3) ---
OUTPUT_FOLDER = NOTEBOOK_DIR / "output_aug_v3"
STATE_FILE    = NOTEBOOK_DIR / "batches_aug_v3.pkl"

# --- Env ---
env_path = NOTEBOOK_DIR.parent / ".env"
load_dotenv(env_path)
HF_TOKEN = os.environ.get("HF_TOKEN", "")
if not HF_TOKEN:
    raise RuntimeError("HF_TOKEN not set in .env")

login(HF_TOKEN)
print("HF login OK | Imports OK")
print(f"OUTPUT_FOLDER: {OUTPUT_FOLDER} (exists={OUTPUT_FOLDER.exists()})")
print(f"STATE_FILE   : {STATE_FILE} (exists={STATE_FILE.exists()})")

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


HF login OK | Imports OK
OUTPUT_FOLDER: output_aug_v3 (exists=True)
STATE_FILE   : batches_aug_v3.pkl (exists=True)


## 1. Load items_tv_v7 từ HF Hub

In [2]:
print(f"Loading {SOURCE_TV7}...")
ds_v7  = load_dataset(SOURCE_TV7)
train_v7 = list(ds_v7["train"])
val_v7   = list(ds_v7["validation"])
test_v7  = list(ds_v7["test"])
print(f"Loaded: train={len(train_v7):,} | val={len(val_v7):,} | test={len(test_v7):,}")

s = train_v7[0]
assert s.get("full"),    "full is None — kiểm tra lại items_tv_v7!"
assert s.get("summary"), "summary is None — kiểm tra lại items_tv_v7!"
print(f"Columns : {list(s.keys())}")
print(f"Sample 0: {s['title'][:80]}")

Loading SeanSunny/items_tv_v7...
Loaded: train=110,000 | val=5,000 | test=5,000
Columns : ['title', 'category', 'price', 'full', 'brand', 'summary']
Sample 0: Pin Tương Thích Cho Laptop Dell Vostro 14 5459 - Hàng Nhập Khẩu New Seal TEEMO P


## 2. Filter + parse_summary

In [3]:
train_filtered = [row for row in train_v7 if row["price"] <= MAX_PRICE]
for idx, row in enumerate(train_filtered):
    row["_idx"] = idx
print(f"train_filtered: {len(train_filtered):,} / {len(train_v7):,}")


def parse_summary(summary: str):
    """Returns (header, body) or None."""
    if not summary:
        return None
    header_lines, body_lines = [], []
    for line in summary.strip().split("\n"):
        line = line.strip()
        if not line:
            continue
        if line.startswith("Tiêu đề:") or line.startswith("Tieu de:"):
            header_lines.append(line)
        elif line.startswith("Danh mục:") or line.startswith("Danh muc:"):
            header_lines.append(line)
        elif line.startswith("Thương hiệu:") or line.startswith("Thuong hieu:"):
            header_lines.append(line)
        elif line.startswith("Mô tả:") or line.startswith("Mo ta:"):
            body_lines.append(line)
        elif line.startswith("Thông số:") or line.startswith("Thong so:"):
            body_lines.append(line)
    if len(header_lines) == 3 and len(body_lines) == 2:
        return "\n".join(header_lines), "\n".join(body_lines)
    return None


ok = sum(1 for row in train_filtered[:2000] if parse_summary(row["summary"]))
print(f"parse_summary check (2000): OK={ok} FAIL={2000-ok}")

train_filtered: 85,727 / 110,000
parse_summary check (2000): OK=1998 FAIL=2


## 3. Price bucket multipliers

In [4]:
BUCKETS = [
    ("<50K",     0,          50_000,   5),
    ("50-100K",  50_000,    100_000,   3),
    ("100-200K", 100_000,   200_000,   2),
    ("200-500K", 200_000,   500_000,   1),
    ("500K-1M",  500_000, 1_000_001,   4),
]

prices = np.array([row["price"] for row in train_filtered])
bucket_multipliers = {}

for name, lo, hi, mult in BUCKETS:
    mask = (prices >= lo) & (prices < hi)
    for idx in np.where(mask)[0]:
        bucket_multipliers[int(idx)] = mult

assert len(bucket_multipliers) == len(train_filtered)
print(f"bucket_multipliers OK: {len(bucket_multipliers):,} items")

bucket_multipliers OK: 85,727 items


## 4. Define AugBatchManager + load state

In [5]:
MODEL      = "openai/gpt-oss-20b"
BATCH_SIZE = 1_000
BATCHES_FOLDER = NOTEBOOK_DIR / "batches_aug_v3"


def build_user_message(full_text: str, body: str) -> str:
    return f"Thông tin sản phẩm gốc:\n{full_text}\n\n---\nTóm tắt hiện tại:\n{body}"


def parse_aug_output(llm_text: str):
    mo_ta = thong_so = None
    for line in llm_text.strip().split("\n"):
        line = line.strip()
        if line.startswith("Mô tả:") or line.startswith("Mo ta:"):
            mo_ta = re.sub(r'^(M[oô] t[aả]):?\s*', '', line).strip()
        elif line.startswith("Thông số:") or line.startswith("Thong so:"):
            thong_so = re.sub(r'^(Th[oô]ng s[oố]):?\s*', '', line).strip()
    if mo_ta and thong_so:
        return mo_ta, thong_so
    return None


@dataclass
class AugBatch:
    start: int
    end: int
    filename: str
    file_id: Optional[str] = None
    batch_id: Optional[str] = None
    output_file_id: Optional[str] = None
    done: bool = False


class AugBatchManager:
    batches: list = []
    request_list: list = []

    @classmethod
    def build_requests(cls, filtered_items, multipliers):
        cls.request_list = []
        skipped = 0
        for idx, row in enumerate(tqdm(filtered_items, desc="Building requests")):
            result = parse_summary(row["summary"])
            if result is None:
                skipped += 1
                continue
            _, body = result
            full_text = row["full"] or ""
            for v in range(multipliers[idx]):
                cls.request_list.append((idx, v, build_user_message(full_text, body)))
        print(f"request_list rebuilt: {len(cls.request_list):,} | skipped: {skipped}")

    @classmethod
    def load(cls):
        with STATE_FILE.open("rb") as f:
            cls.batches = pickle.load(f)
        print(f"State loaded: {len(cls.batches)} batches")
        done_count = sum(1 for b in cls.batches if b.done)
        print(f"  Done: {done_count} / {len(cls.batches)}")


print("Classes defined OK")

Classes defined OK


In [6]:
# Load batch state từ pkl
AugBatchManager.load()

# Rebuild request_list (cần để biết mapping item_idx → version)
AugBatchManager.build_requests(train_filtered, bucket_multipliers)

State loaded: 186 batches
  Done: 186 / 186


Building requests:   0%|          | 0/85727 [00:00<?, ?it/s]

request_list rebuilt: 185,584 | skipped: 43


## 5. Đọc output files local → rebuild aug_results

In [7]:
# Kiểm tra output files có đủ không
output_files = list(OUTPUT_FOLDER.glob("aug_*.jsonl"))
print(f"Output files found: {len(output_files)} / {len(AugBatchManager.batches)} batches")

missing_files = [
    b.filename for b in AugBatchManager.batches
    if not (OUTPUT_FOLDER / b.filename).exists()
]
if missing_files:
    print(f"WARNING: {len(missing_files)} missing files: {missing_files[:5]}")
else:
    print("All output files present OK")

Output files found: 186 / 186 batches
All output files present OK


In [8]:
aug_results = {}  # (item_idx, version) -> llm_text

for batch in tqdm(AugBatchManager.batches, desc="Reading outputs"):
    out_path = OUTPUT_FOLDER / batch.filename
    if not out_path.exists():
        print(f"WARNING: missing {batch.filename}")
        continue
    with out_path.open(encoding="utf-8") as f:
        for line in f:
            obj = json.loads(line)
            cid = obj["custom_id"]
            item_idx, version = int(cid.split("_")[0]), int(cid.split("_")[1])
            aug_results[(item_idx, version)] = (
                obj["response"]["body"]["choices"][0]["message"]["content"]
            )

print(f"LLM outputs collected: {len(aug_results):,} / {len(AugBatchManager.request_list):,}")
print(f"Missing: {len(AugBatchManager.request_list) - len(aug_results)}")

Reading outputs:   0%|          | 0/186 [00:00<?, ?it/s]

LLM outputs collected: 185,583 / 185,584
Missing: 1


## 6. Build v8_rows

In [9]:
v8_rows = []
build_ok = build_fail = 0

for (item_idx, version), llm_text in tqdm(aug_results.items(), desc="Building v8 rows"):
    row    = train_filtered[item_idx]
    result = parse_summary(row["summary"])
    if result is None:
        build_fail += 1
        continue
    header, _ = result
    parsed = parse_aug_output(llm_text)
    if parsed is None:
        build_fail += 1
        continue
    mo_ta, thong_so = parsed
    new_body   = f"Mô tả: {mo_ta}\nThông số: {thong_so}"
    summary_v2 = f"{header}\n{new_body}"
    v8_rows.append({
        "title":            row["title"],
        "category":         row["category"],
        "price":            row["price"],
        "full":             row["full"],
        "brand":            row.get("brand"),
        "summary":          row["summary"],
        "summary_version2": summary_v2,
        "aug_version":      version,
    })
    build_ok += 1

print(f"v8 rows built: {build_ok:,} OK | {build_fail} failed")
if v8_rows:
    print(f"Sample summary_version2:\n{v8_rows[0]['summary_version2']}")

Building v8 rows:   0%|          | 0/185583 [00:00<?, ?it/s]

v8 rows built: 183,385 OK | 2198 failed
Sample summary_version2:
Tiêu đề: 40 Viên Pin Maxell AAA Than (Carbon)
Danh mục: Pin & Điện Lực
Thương hiệu: Maxell
Mô tả: Pin carbon Maxell AAA 40 viên, giá thành thấp, thích hợp cho đồ chơi, thiết bị gia đình và công nghệ.
Thông số: Dung tích 1,5V, kích thước 42mm x10mm, thời gian lưu trữ 3 năm.


## 7. Push items_tv_v8

In [10]:
# Giải phóng RAM trước khi build Dataset — tránh OOM trên WSL
del train_v7, aug_results
import gc
gc.collect()
print("Memory freed: train_v7, aug_results deleted")

Memory freed: train_v7, aug_results deleted


In [ ]:
from datasets import Features, Value

V8_FEATURES = Features({
    "title":            Value("string"),
    "category":         Value("string"),
    "price":            Value("int64"),
    "brand":            Value("string"),
    "summary":          Value("string"),
    "summary_version2": Value("string"),
    "aug_version":      Value("int64"),
})

def strip_full(rows):
    return [{k: v for k, v in row.items() if k != "full"} for row in rows]

def add_v2_col(rows):
    return [{**{k: v for k, v in row.items() if k != "full"},
             "summary_version2": None, "aug_version": None} for row in rows]

ds_v8 = DatasetDict({
    "train":      Dataset.from_list(strip_full(v8_rows),    features=V8_FEATURES),
    "validation": Dataset.from_list(add_v2_col(val_v7),     features=V8_FEATURES),
    "test":       Dataset.from_list(add_v2_col(test_v7),    features=V8_FEATURES),
})
print(ds_v8)
print(f"Train columns: {ds_v8['train'].column_names}")
print(f"Pushing {OUTPUT_TV8}...")
ds_v8.push_to_hub(OUTPUT_TV8, private=True)
print(f"Pushed: https://huggingface.co/datasets/{OUTPUT_TV8}")

## 8. Build items_prompts_tv_4

- Train: 85K gốc (`items_prompts_tv_3`) + ~183K aug (`summary_version2`)
- Val / Test: giữ nguyên từ `items_prompts_tv_3`

In [ ]:
print(f"Loading {SOURCE_TV3}...")
ds_tv3 = load_dataset(SOURCE_TV3)
orig_train = list(ds_tv3["train"])
orig_val   = list(ds_tv3["val"])
orig_test  = list(ds_tv3["test"])
print(f"orig train={len(orig_train):,} | val={len(orig_val):,} | test={len(orig_test):,}")
assert set(orig_train[0].keys()) == {"prompt", "completion", "price_vnd_true"}
print("Schema OK")

In [ ]:
aug_examples = []
for row in tqdm(v8_rows, desc="Building prompts tv4"):
    sv2 = row["summary_version2"]
    if not sv2:
        continue
    aug_examples.append({
        "prompt":         f"{QUESTION_FULL}\n{sv2}{PRICE_PREF}",
        "completion":     str(int(round(row["price"] / 1000))),
        "price_vnd_true": int(row["price"]),
    })

print(f"Augmented prompts: {len(aug_examples):,}")

combined_train = orig_train + aug_examples
random.seed(SEED)
random.shuffle(combined_train)

print(f"Combined train: {len(orig_train):,} (orig) + {len(aug_examples):,} (aug) = {len(combined_train):,}")

empty_p = sum(1 for ex in combined_train if not ex["prompt"])
empty_c = sum(1 for ex in combined_train if not ex["completion"])
assert empty_p == 0 and empty_c == 0
prices_s = [ex["price_vnd_true"] for ex in combined_train]
print(f"Price range: {min(prices_s):,} — {max(prices_s):,} VND")
print("Quality check OK")

In [ ]:
ds_tv4 = DatasetDict({
    "train": Dataset.from_list(combined_train),
    "val":   Dataset.from_list(orig_val),
    "test":  Dataset.from_list(orig_test),
})
print(ds_tv4)
print(f"Pushing {OUTPUT_TV4}...")
ds_tv4.push_to_hub(OUTPUT_TV4, private=True)
print(f"Pushed: https://huggingface.co/datasets/{OUTPUT_TV4}")
print(f"\nDone! Train size: {len(combined_train):,} (~{len(combined_train)/1000:.0f}K)")

## Summary

| Dataset | Split | Rows | Note |
|---|---|---|---|
| `items_tv_v8` | train | ~183K | augmented rows, cột `summary_version2` |
| `items_tv_v8` | val/test | 5K/5K | giữ v7, `summary_version2=None` |
| `items_prompts_tv_4` | train | ~268K | 85K orig + 183K aug |
| `items_prompts_tv_4` | val/test | 3,926/3,872 | giữ tv_3 |

**Bước tiếp:** Chạy `06_train_v4_scratch.ipynb` trên RTX 5090 với `items_prompts_tv_4`.